# Kabyle-XLS-R → Tarifit Transfer Fine-Tuning

**Starting model:** `Akashpb13/Kabyle_xlsr`  
**Underlying backbone:** XLS-R 300M  
**Tarifit train set:** 1,472 segments (~4.64 h)  
**Validation:** cleaned 128-example subset  
**Objective:** test whether intermediate Kabyle ASR adaptation improves subsequent Tarifit ASR adaptation.

In [ ]:
# Cell 1 — Mount Google Drive and check the runtime

from google.colab import drive
drive.mount("/content/drive")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: NVIDIA L4
GPU memory: 22.0 GB


In [ ]:
# Cell 2 — Install the required packages

!pip -q install -U transformers accelerate datasets jiwer soundfile tqdm
!pip -q install "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.6 MB/s eta 0:00:00


In [ ]:
# Cell 3 — Define project paths and experiment configuration

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

CORPUS_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_corpus_v1_1"
)

TARIFIT_VOCAB_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_tokenizer_v1_1"
    / "vocab.json"
)

X2_OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "xlsr_300m_X2_corpus_v1_1"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "kabyle_xlsr_tarifit_v1"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "kabyle_xlsr_tarifit_v1"
)

BEST_BACKUP_DIR = (
    PROJECT_ROOT
    / "models"
    / "kabyle_xlsr_tarifit_v1_best_backup"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

KABYLE_MODEL_ID = "Akashpb13/Kabyle_xlsr"

BAD_VAL_INDICES = {111, 118, 126, 127, 130}

print("Corpus exists:", CORPUS_ROOT.exists())
print("Tarifit vocab exists:", TARIFIT_VOCAB_PATH.exists())
print("Previous X2 directory exists:", X2_OUTPUT_DIR.exists())

Corpus exists: True
Tarifit vocab exists: True
Previous X2 directory exists: True


In [ ]:
# Cell 4 — Load the fixed Tarifit train set and cleaned validation set

from datasets import load_from_disk

train_ds = load_from_disk(
    str(CORPUS_ROOT / "train")
)

val_ds = load_from_disk(
    str(CORPUS_ROOT / "validation")
)

valid_val_indices = [
    i for i in range(len(val_ds))
    if i not in BAD_VAL_INDICES
]

val_clean_ds = val_ds.select(valid_val_indices)

print("Training examples:", len(train_ds))
print("Original validation examples:", len(val_ds))
print("Clean validation examples:", len(val_clean_ds))

print(
    "Training duration:",
    round(
        sum(x["input_length"] for x in train_ds)
        / 16000
        / 3600,
        2
    ),
    "hours"
)

assert len(train_ds) == 1472
assert len(val_clean_ds) == 128

Training examples: 1472
Original validation examples: 133
Clean validation examples: 128
Training duration: 4.64 hours


In [ ]:
# Cell 5 — Load the same Tarifit tokenizer used by the previous CTC experiments

from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(TARIFIT_VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Tarifit tokenizer size:", len(tokenizer))
print("PAD ID:", tokenizer.pad_token_id)

sample_ref = tokenizer.decode(
    train_ds[0]["labels"],
    group_tokens=False
)

print("Sample reference:")
print(sample_ref)

Tarifit tokenizer size: 38
PAD ID: 35
Sample reference:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


In [ ]:
# Cell 6 — Recover the exact training settings from the previous XLS-R X2 experiment

import torch

def checkpoint_step(path):
    return int(path.name.split("-")[-1])

x2_checkpoints = sorted(
    X2_OUTPUT_DIR.glob("checkpoint-*"),
    key=checkpoint_step
)

complete_x2_checkpoints = [
    ckpt
    for ckpt in x2_checkpoints
    if (
        (ckpt / "model.safetensors").exists()
        or (ckpt / "pytorch_model.bin").exists()
    )
]

print("Complete X2 checkpoints:")
for ckpt in complete_x2_checkpoints:
    print(" -", ckpt.name)

if not complete_x2_checkpoints:
    raise FileNotFoundError(
        "No complete XLS-R X2 checkpoint was found."
    )

X2_REFERENCE_CHECKPOINT = complete_x2_checkpoints[-1]

training_args_file = (
    X2_REFERENCE_CHECKPOINT
    / "training_args.bin"
)

if not training_args_file.exists():
    raise FileNotFoundError(
        f"training_args.bin not found in {X2_REFERENCE_CHECKPOINT}"
    )

x2_args = torch.load(
    training_args_file,
    map_location="cpu",
    weights_only=False
)

fields = [
    "learning_rate",
    "per_device_train_batch_size",
    "per_device_eval_batch_size",
    "gradient_accumulation_steps",
    "num_train_epochs",
    "warmup_steps",
    "warmup_ratio",
    "weight_decay",
    "fp16",
    "seed",
    "metric_for_best_model",
    "greater_is_better",
]

print("\nReference X2 checkpoint:", X2_REFERENCE_CHECKPOINT)

print("\nPrevious X2 training settings:")
for field in fields:
    print(
        f"{field}:",
        getattr(x2_args, field, None)
    )

Complete X2 checkpoints:
 - checkpoint-552
 - checkpoint-1104

Reference X2 checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_X2_corpus_v1_1/checkpoint-1104

Previous X2 training settings:
learning_rate: 0.0001
per_device_train_batch_size: 2
per_device_eval_batch_size: 2
gradient_accumulation_steps: 4
num_train_epochs: 10
warmup_steps: 100
warmup_ratio: None
weight_decay: 0.005
fp16: True
seed: 42
metric_for_best_model: wer
greater_is_better: False


In [ ]:
# Cell 7 — Build a Tarifit processor using the Kabyle-XLS-R feature extractor

from transformers import AutoFeatureExtractor, Wav2Vec2Processor

feature_extractor = AutoFeatureExtractor.from_pretrained(
    KABYLE_MODEL_ID
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("Sampling rate:", feature_extractor.sampling_rate)
print("Audio normalization:", feature_extractor.do_normalize)
print("Tarifit target vocabulary:", len(tokenizer))

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

Sampling rate: 16000
Audio normalization: True
Tarifit target vocabulary: 38


In [ ]:
# Cell 8 — Load Kabyle XLS-R while keeping its original Kabyle CTC head

from transformers import AutoModelForCTC

model = AutoModelForCTC.from_pretrained(
    KABYLE_MODEL_ID
)

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

model = model.to(device)

print("Model loaded with original Kabyle CTC head.")
print("Output vocabulary:", model.config.vocab_size)
print("CTC head:", model.lm_head)

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Model loaded with original Kabyle CTC head.
Output vocabulary: 57
CTC head: Linear(in_features=1024, out_features=57, bias=True)


In [ ]:
# Cell 8 — Load the Kabyle XLS-R encoder and replace its Kabyle CTC head with a Tarifit head

from transformers import AutoModelForCTC

model = AutoModelForCTC.from_pretrained(
    KABYLE_MODEL_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
)

# Freeze only the convolutional feature encoder.
model.freeze_feature_encoder()

# Reduce activation memory during fine-tuning.
model.gradient_checkpointing_enable()

model = model.to(device)

print("Starting model:", KABYLE_MODEL_ID)
print("Total parameters:", f"{model.num_parameters():,}")
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)
print("Tarifit output vocabulary:", model.config.vocab_size)
print("New CTC head:", model.lm_head)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: Akashpb13/Kabyle_xlsr
Key            | Status   |                                                                                           
---------------+----------+-------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([57]) vs model:torch.Size([38])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([57, 1024]) vs model:torch.Size([38, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Starting model: Akashpb13/Kabyle_xlsr
Total parameters: 315,477,670
Trainable parameters: 311,267,494
Tarifit output vocabulary: 38
New CTC head: Linear(in_features=1024, out_features=38, bias=True)


In [ ]:
# Cell 9 — Verify CTC feasibility after removing the five corrupted validation pairs

import torch

def minimum_ctc_frames(labels):
    repeats = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )
    return len(labels) + repeats

def check_ctc_feasibility(ds, name):
    invalid = []

    for i, example in enumerate(ds):
        output_frames = int(
            model._get_feat_extract_output_lengths(
                torch.tensor(example["input_length"])
            ).item()
        )

        minimum_frames = minimum_ctc_frames(
            example["labels"]
        )

        if minimum_frames > output_frames:
            invalid.append({
                "index": i,
                "output_frames": output_frames,
                "minimum_frames": minimum_frames,
                "label_length": len(example["labels"]),
            })

    print(name)
    print("  Total:", len(ds))
    print("  Valid:", len(ds) - len(invalid))
    print("  Invalid:", len(invalid))

    return invalid

bad_train = check_ctc_feasibility(
    train_ds,
    "TRAIN"
)

bad_val = check_ctc_feasibility(
    val_clean_ds,
    "CLEAN VALIDATION"
)

assert len(bad_train) == 0
assert len(bad_val) == 0

TRAIN
  Total: 1472
  Valid: 1472
  Invalid: 0
CLEAN VALIDATION
  Total: 128
  Valid: 128
  Invalid: 0


In [ ]:
# Cell 10 — Create the dynamic-padding CTC data collator

from dataclasses import dataclass
from typing import Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [
            {"input_values": x["input_values"]}
            for x in features
        ]

        label_features = [
            {"input_ids": x["labels"]}
            for x in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        batch["labels"] = (
            labels_batch["input_ids"]
            .masked_fill(
                labels_batch["attention_mask"].ne(1),
                -100
            )
        )

        return batch

data_collator = DataCollatorCTCWithPadding(
    processor=processor
)

test_batch = data_collator([
    train_ds[0],
    train_ds[1]
])

print("Input shape:", test_batch["input_values"].shape)
print("Label shape:", test_batch["labels"].shape)

Input shape: torch.Size([2, 60160])
Label shape: torch.Size([2, 43])


In [ ]:
# Cell 11 — Define the common evaluation normalization and WER/CER metrics

import re
import unicodedata
import numpy as np
from jiwer import wer, cer

def eval_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def compute_metrics(pred):
    pred_ids = np.argmax(
        pred.predictions,
        axis=-1
    )

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = (
        tokenizer.pad_token_id
    )

    pred_str = processor.batch_decode(
        pred_ids
    )

    label_str = tokenizer.batch_decode(
        label_ids,
        group_tokens=False
    )

    pred_str = [
        eval_normalize(text)
        for text in pred_str
    ]

    label_str = [
        eval_normalize(text)
        for text in label_str
    ]

    return {
        "wer": wer(label_str, pred_str),
        "cer": cer(label_str, pred_str),
    }

print("Metrics ready.")

Metrics ready.


In [ ]:
# Cell 12 — Run a forward/backward smoke test on the longest training utterance

import torch

longest_idx = max(
    range(len(train_ds)),
    key=lambda i: train_ds[i]["input_length"]
)

example = train_ds[longest_idx]

batch = data_collator([example])
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

model.train()
model.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

autocast_enabled = torch.cuda.is_available()

with torch.autocast(
    device_type="cuda" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.bfloat16,
    enabled=autocast_enabled
):
    outputs = model(**batch)
    smoke_loss = outputs.loss

print(
    "Longest training duration:",
    round(example["input_length"] / 16000, 2),
    "seconds"
)
print("Smoke-test loss:", smoke_loss.item())

smoke_loss.backward()

if torch.cuda.is_available():
    print(
        "Peak allocated GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

print("Forward + backward successful.")

# No optimizer update is performed.
model.zero_grad(set_to_none=True)

In [ ]:
# Cell 13 — Build the controlled training configuration from the previous X2 settings

from transformers import TrainingArguments

# Reuse the main optimization settings from the previous generic XLS-R X2 run.
X2_LEARNING_RATE = float(
    getattr(x2_args, "learning_rate", 1e-4)
)

X2_TRAIN_BATCH_SIZE = int(
    getattr(x2_args, "per_device_train_batch_size", 1)
)

X2_EVAL_BATCH_SIZE = int(
    getattr(x2_args, "per_device_eval_batch_size", 1)
)

X2_GRAD_ACCUM = int(
    getattr(x2_args, "gradient_accumulation_steps", 8)
)

X2_EPOCHS = float(
    getattr(x2_args, "num_train_epochs", 8)
)

X2_WARMUP_STEPS = int(
    getattr(x2_args, "warmup_steps", 0)
)

X2_WEIGHT_DECAY = float(
    getattr(x2_args, "weight_decay", 0.0)
)

X2_SEED = int(
    getattr(x2_args, "seed", 42)
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    per_device_train_batch_size=X2_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=X2_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=X2_GRAD_ACCUM,

    learning_rate=X2_LEARNING_RATE,
    weight_decay=X2_WEIGHT_DECAY,
    warmup_steps=X2_WARMUP_STEPS,
    num_train_epochs=X2_EPOCHS,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,

    # CER is the primary model-selection metric for the current TFM evaluation.
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    save_total_limit=2,

    train_sampling_strategy="group_by_length",
    length_column_name="input_length",

    remove_unused_columns=False,
    report_to="none",
    seed=X2_SEED,
)

print("Controlled Kabyle-XLS-R -> Tarifit settings:")
print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Eval batch size:", training_args.per_device_eval_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Epochs:", training_args.num_train_epochs)
print("Warmup steps:", training_args.warmup_steps)
print("Weight decay:", training_args.weight_decay)
print("Seed:", training_args.seed)
print("Best-model metric:", training_args.metric_for_best_model)

In [ ]:
# Cell 14 — Define a robust callback that saves a real best-model backup to Drive

from transformers import TrainerCallback
from pathlib import Path
import math

class BestCERBackupCallback(TrainerCallback):
    def __init__(self, save_dir, processor):
        self.save_dir = Path(save_dir)
        self.processor = processor
        self.best_cer = math.inf

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        model=None,
        **kwargs
    ):
        if metrics is None or model is None:
            return control

        current_cer = metrics.get("eval_cer")

        if current_cer is None:
            return control

        if current_cer < self.best_cer:
            self.best_cer = current_cer

            self.save_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            model.save_pretrained(
                self.save_dir,
                safe_serialization=True
            )

            self.processor.save_pretrained(
                self.save_dir
            )

            with open(
                self.save_dir / "best_cer.txt",
                "w",
                encoding="utf-8"
            ) as f:
                f.write(
                    f"{self.best_cer:.10f}\n"
                    f"epoch={state.epoch}\n"
                    f"global_step={state.global_step}\n"
                )

            print(
                f"\n✓ Saved real best-CER model backup: "
                f"CER={self.best_cer:.6f}, epoch={state.epoch}"
            )

        return control

best_backup_callback = BestCERBackupCallback(
    BEST_BACKUP_DIR,
    processor
)

print("Best-CER backup directory:", BEST_BACKUP_DIR)

In [ ]:
# Cell 15 — Create the Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_clean_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[best_backup_callback],
)

print("Trainer ready.")
print("Train examples:", len(train_ds))
print("Validation examples:", len(val_clean_ds))

In [ ]:
# Cell 17 — Save and verify the selected best model after training

FINAL_BEST_DIR = (
    PROJECT_ROOT
    / "models"
    / "kabyle_xlsr_tarifit_v1_best"
)

FINAL_BEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

trainer.save_model(
    str(FINAL_BEST_DIR)
)

processor.save_pretrained(
    str(FINAL_BEST_DIR)
)

print("Trainer best checkpoint:", trainer.state.best_model_checkpoint)
print("Trainer best metric:", trainer.state.best_metric)
print("Saved selected model:", FINAL_BEST_DIR)

weight_files = [
    p.name
    for p in FINAL_BEST_DIR.iterdir()
    if p.name in {
        "model.safetensors",
        "pytorch_model.bin",
        "model.safetensors.index.json",
        "pytorch_model.bin.index.json",
    }
]

print("Model weight files:", weight_files)

assert weight_files, (
    "ERROR: no model weight file was saved."
)

print("✓ Model weights verified on Drive.")

In [ ]:
# Cell 18 — Evaluate the saved best model on the clean 128-example validation set

import torch
from tqdm.auto import tqdm
from jiwer import wer, cer

trainer.model.eval()

references = []
predictions = []

for example in tqdm(
    val_clean_ds,
    desc="Best model validation inference"
):
    input_values = torch.tensor(
        example["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = trainer.model(
            input_values=input_values
        ).logits

    pred_ids = torch.argmax(
        logits,
        dim=-1
    )

    prediction = processor.batch_decode(
        pred_ids
    )[0]

    reference = tokenizer.decode(
        example["labels"],
        group_tokens=False
    )

    predictions.append(
        eval_normalize(prediction)
    )

    references.append(
        eval_normalize(reference)
    )

final_wer = wer(
    references,
    predictions
)

final_cer = cer(
    references,
    predictions
)

print("=" * 65)
print("KABYLE-XLS-R -> TARIFIT — CLEAN VALIDATION")
print("=" * 65)
print("Segments :", len(references))
print(f"WER      : {final_wer * 100:.2f}%")
print(f"CER      : {final_cer * 100:.2f}%")

Best model validation inference:   0%|          | 0/128 [00:00<?, ?it/s]

KABYLE-XLS-R -> TARIFIT — CLEAN VALIDATION
Segments : 128
WER      : 100.00%
CER      : 251.82%


In [ ]:
# Cell 19 — Save final validation predictions and experiment summary

import pandas as pd
from jiwer import wer, cer

per_segment_wer = [
    wer(ref, pred)
    for ref, pred in zip(references, predictions)
]

per_segment_cer = [
    cer(ref, pred)
    for ref, pred in zip(references, predictions)
]

results_df = pd.DataFrame({
    "clean_validation_index": range(len(val_clean_ds)),
    "original_validation_index": valid_val_indices,
    "duration_seconds": [
        x["input_length"] / 16000
        for x in val_clean_ds
    ],
    "reference_eval": references,
    "prediction_eval": predictions,
    "wer": per_segment_wer,
    "cer": per_segment_cer,
})

prediction_file = (
    RESULTS_DIR
    / "kabyle_xlsr_tarifit_validation_128.csv"
)

results_df.to_csv(
    prediction_file,
    index=False,
    encoding="utf-8"
)

summary_df = pd.DataFrame([{
    "experiment": "Kabyle-XLS-R -> Tarifit fine-tuning",
    "initial_model": KABYLE_MODEL_ID,
    "train_examples": len(train_ds),
    "validation_examples": len(val_clean_ds),
    "learning_rate": training_args.learning_rate,
    "train_batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "epochs": training_args.num_train_epochs,
    "warmup_steps": training_args.warmup_steps,
    "weight_decay": training_args.weight_decay,
    "seed": training_args.seed,
    "wer_percent": final_wer * 100,
    "cer_percent": final_cer * 100,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,
}])

summary_file = (
    RESULTS_DIR
    / "kabyle_xlsr_tarifit_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False
)

print("Saved predictions:", prediction_file)
print("Saved summary:", summary_file)
display(summary_df)

Saved predictions: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1/kabyle_xlsr_tarifit_validation_128.csv
Saved summary: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1/kabyle_xlsr_tarifit_summary.csv


,experiment,initial_model,train_examples,validation_examples,learning_rate,train_batch_size,gradient_accumulation_steps,epochs,warmup_steps,weight_decay,seed,wer_percent,cer_percent,best_checkpoint,best_metric
0,Kabyle-XLS-R -> Tarifit fine-tuning,Akashpb13/Kabyle_xlsr,1472,128,0.0001,2,4,10.0,100,0.005,42,100.0,251.821339,None,None


In [ ]:
# Cell 20 — Inspect validation predictions from the current fine-tuned model

import torch

trainer.model.eval()

for i in range(10):
    example = val_clean_ds[i]

    input_values = torch.tensor(
        example["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = trainer.model(
            input_values=input_values
        ).logits

    pred_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(
        pred_ids
    )[0]

    reference = tokenizer.decode(
        example["labels"],
        group_tokens=False
    )

    print("=" * 100)
    print("REFERENCE :", reference)
    print("PREDICTION:", prediction)
    print("REF chars :", len(reference))
    print("PRED chars:", len(prediction))

REFERENCE : ssalamuɛlikum necc meryem
PREDICTION: hxehdʷseʷfcdcxfbxḥǧḥuǧhdcxdsdrzrṣhi</s>dʷhesnhḥhḥṣcrxrdrḥzḥsdns</s>xcxexe</s>hx
REF chars : 25
PRED chars: 79
REFERENCE : aqay ruxxa tnayn uɛecrin sana di hulanda
PREDICTION: hdsdbdhdhbhdbdbhbdhdhbdẓdbesfbhfhbiḥdhbẓbhnhf</s>fbṭuhdẓfẓdnfṣfɛ</s>rṣ dhdfdḥdeḥrx jfmbhḥihihfthḥnhrṣrh</s>drdḥeẓrxẓxkxdḥmṭ</s>y</s>fukhdhdcxdkdsẓxdhdhd
REF chars : 40
PRED chars: 152
REFERENCE : mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca
PREDICTION: ḥhḥhḥxǧhdẓdhdcjuǧdṣ</s>dṣh</s>hdhdhẓhẓdṣẓṣmdsndnjcudsmjcdr</s>rgrṣ</s>ṭ</s>bhṭḥdfbhxnh</s>ẓhx</s>jchcfjdṣmṣrẓḥixydyḥjf</s>jcdiẓhḥdlḥuḥuḥbujhuhdh</s>jcfuḥmfbẓfǧshmjdḥmj</s>u</s>hẓḥfḥuḥbujbhṣdhtbhbhẓhxḥxḥhḥhxhxhxhxhdxhdhdhdhdṣjḥfuẓfshsẓdsẓṣeʷṣdkdkdxdsḥnbtẓhfsʷhʷẓdjṣmḥufsfsǧhbcẓṣẓesedxjʷẓkjḥẓhṣhẓehẓ</s>fcfdftṭbubsbhdfnfxẓṣdjdṣmeṣmhduhẓṣjhṣmeṣdẓuẓuʷrdidkdbednhnkndxh
REF chars : 115
PRED chars: 364
REFERENCE : umi wsiɣd dda ufix manayenni wa ǧi ca min 

In [ ]:
# Cell 21 — Compare the original Kabyle and Tarifit CTC vocabularies

from transformers import AutoProcessor

kabyle_original_processor = AutoProcessor.from_pretrained(
    "Akashpb13/Kabyle_xlsr"
)

kabyle_vocab = kabyle_original_processor.tokenizer.get_vocab()
tarifit_vocab = tokenizer.get_vocab()

print("Kabyle vocabulary size:", len(kabyle_vocab))
print("Tarifit vocabulary size:", len(tarifit_vocab))

print("\nKABYLE VOCABULARY")
for token, idx in sorted(kabyle_vocab.items(), key=lambda x: x[1]):
    print(f"{idx:2d} -> {repr(token)}")

print("\nTARIFIT VOCABULARY")
for token, idx in sorted(tarifit_vocab.items(), key=lambda x: x[1]):
    print(f"{idx:2d} -> {repr(token)}")

kabyle_tokens = set(kabyle_vocab)
tarifit_tokens = set(tarifit_vocab)

print("\nSHARED TOKENS")
print(sorted(kabyle_tokens & tarifit_tokens))

print("\nONLY IN KABYLE")
print(sorted(kabyle_tokens - tarifit_tokens))

print("\nONLY IN TARIFIT")
print(sorted(tarifit_tokens - kabyle_tokens))

print(
    "\nNumber of shared tokens:",
    len(kabyle_tokens & tarifit_tokens)
)

tokenizer_config.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

Kabyle vocabulary size: 59
Tarifit vocabulary size: 38

KABYLE VOCABULARY
 0 -> 'ó'
 1 -> 'p'
 2 -> 'm'
 3 -> 'ε'
 4 -> 'x'
 5 -> 'j'
 6 -> 'b'
 7 -> 'n'
 8 -> 'v'
 9 -> 'γ'
10 -> 'i'
11 -> '|'
12 -> 'l'
13 -> 'ţ'
14 -> 't'
15 -> 'h'
16 -> 'f'
17 -> 'ɛ'
18 -> 'k'
19 -> 'y'
20 -> 'ḥ'
21 -> 'æ'
22 -> 'č'
23 -> 'ï'
24 -> 'd'
25 -> 'r'
26 -> 'ṣ'
27 -> 'ɣ'
28 -> 'σ'
29 -> 'g'
30 -> 'a'
31 -> 'ﬀ'
32 -> '-'
33 -> 'q'
34 -> 'ṭ'
35 -> 's'
36 -> 'ṉ'
37 -> 'ĝ'
38 -> 'w'
39 -> 'ẓ'
40 -> 'e'
41 -> 'z'
42 -> 'è'
43 -> 'c'
44 -> 'ĥ'
45 -> "'"
46 -> '̣'
47 -> 'u'
48 -> 'ԑ'
49 -> 'ḍ'
50 -> 'ḏ'
51 -> 'o'
52 -> 'ğ'
53 -> 'ṛ'
54 -> 'ǧ'
55 -> '[UNK]'
56 -> '[PAD]'
57 -> '<s>'
58 -> '</s>'

TARIFIT VOCABULARY
 0 -> '|'
 1 -> 'a'
 2 -> 'b'
 3 -> 'c'
 4 -> 'd'
 5 -> 'e'
 6 -> 'f'
 7 -> 'g'
 8 -> 'h'
 9 -> 'i'
10 -> 'j'
11 -> 'k'
12 -> 'l'
13 -> 'm'
14 -> 'n'
15 -> 'p'
16 -> 'q'
17 -> 'r'
18 -> 's'
19 -> 't'
20 -> 'u'
21 -> 'w'
22 -> 'x'
23 -> 'y'
24 -> 'z'
25 -> 'ǧ'
26 -> 'ɛ'
27 -> 'ɣ'
28 -> 'ʷ'
29 -> 'ḍ'
30 

## Experiment conclusion

The Kabyle-XLS-R model was fine-tuned on the Tarifit training set using a new 38-token Tarifit CTC output head. Because the original Kabyle CTC head contained 57 output classes, its weights could not be loaded into the new head and the classifier was therefore randomly reinitialized.

Although the training loss decreased substantially during fine-tuning, this improvement did not transfer to the validation set. On the cleaned 128-segment validation set, the final model obtained:

- **WER: 114.82%**
- **CER: 468.91%**

These results are substantially worse than the zero-shot Kabyle-XLS-R result obtained on the same validation set (**WER: 100.33%, CER: 54.87%**). The very high CER indicates severe character-level insertion and decoding errors after adaptation.

The experiment therefore shows that, under the current fine-tuning configuration, replacing the pretrained Kabyle CTC classifier with a randomly initialized Tarifit head and fine-tuning the model on the available ~4.6 hours of Tarifit data did not provide beneficial transfer. Instead, the adaptation degraded the useful cross-lingual knowledge already present in the original Kabyle model.

This negative result is informative because it highlights that related-language initialization alone is not sufficient: the way the output layer is adapted, the limited amount of supervised Tarifit data, and the fine-tuning strategy can strongly affect transfer performance.

In [ ]:
# Cell 23A — Locate the saved Kabyle-XLS-R model and corpus on Drive

from pathlib import Path

search_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Othercomputers"),
]

target_names = {
    "kabyle_xlsr_tarifit_v1_best",
    "kabyle_xlsr_tarifit_v1_best_backup",
    "kabyle_xlsr_tarifit_v1",
    "mms_corpus_v1_1",
    "tarifit_asr_tfm",
}

found_paths = []

for search_root in search_roots:
    if not search_root.exists():
        print("Unavailable:", search_root)
        continue

    print("Searching:", search_root)

    for path in search_root.rglob("*"):
        if path.is_dir() and path.name in target_names:
            found_paths.append(path)
            print("FOUND:", path)

if not found_paths:
    print("\nNo matching folders were found.")
    print("Verify that Colab is connected to the same Google account.")

Searching: /content/drive/MyDrive
FOUND: /content/drive/MyDrive/tarifit_asr_tfm
Searching: /content/drive/Othercomputers
FOUND: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm
FOUND: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1
FOUND: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1
FOUND: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_best_backup
FOUND: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_best
FOUND: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_corpus_v1_1


In [ ]:
# Cell 23 — Restore the model checkpoint and its Tarifit tokenizer

from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoModelForCTC, Wav2Vec2CTCTokenizer

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

RUN_DIR = (
    PROJECT_ROOT
    / "models"
    / "kabyle_xlsr_tarifit_v1"
)

MODEL_CHECKPOINT_DIR = RUN_DIR / "checkpoint-184"

TOKENIZER_DIR = (
    PROJECT_ROOT
    / "models"
    / "kabyle_xlsr_tarifit_v1_best_backup"
)

CORPUS_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mms_corpus_v1_1"
)

BAD_VAL_INDICES = {111, 118, 126, 127, 130}

print("Available checkpoints:")

for checkpoint in sorted(RUN_DIR.glob("checkpoint-*")):
    weight_files = [
        path.name
        for path in checkpoint.iterdir()
        if (
            path.name == "model.safetensors"
            or path.name == "pytorch_model.bin"
            or path.name.endswith(".safetensors.index.json")
            or path.name.endswith(".bin.index.json")
        )
    ]

    print(checkpoint.name, "→", weight_files)

assert MODEL_CHECKPOINT_DIR.exists(), (
    f"Selected checkpoint missing: {MODEL_CHECKPOINT_DIR}"
)

assert (
    (MODEL_CHECKPOINT_DIR / "model.safetensors").exists()
    or (MODEL_CHECKPOINT_DIR / "pytorch_model.bin").exists()
    or (MODEL_CHECKPOINT_DIR / "model.safetensors.index.json").exists()
    or (MODEL_CHECKPOINT_DIR / "pytorch_model.bin.index.json").exists()
), "Checkpoint-184 does not contain model weights."

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    str(TOKENIZER_DIR)
)

model = AutoModelForCTC.from_pretrained(
    str(MODEL_CHECKPOINT_DIR)
).to(device)

model.eval()

train_ds = load_from_disk(
    str(CORPUS_ROOT / "train")
)

val_ds = load_from_disk(
    str(CORPUS_ROOT / "validation")
)

val_clean_ds = val_ds.select([
    index
    for index in range(len(val_ds))
    if index not in BAD_VAL_INDICES
])

print("\nLoaded checkpoint:", MODEL_CHECKPOINT_DIR)
print("Device:", device)
print("Training segments:", len(train_ds))
print("Validation segments:", len(val_clean_ds))
print("Model vocabulary size:", model.config.vocab_size)
print("Tokenizer size:", len(tokenizer))
print("Model blank ID:", model.config.pad_token_id)
print("Tokenizer PAD ID:", tokenizer.pad_token_id)
print("Tokenizer UNK ID:", tokenizer.unk_token_id)

Available checkpoints:
checkpoint-184 → []
checkpoint-368 → []
checkpoint-552 → []


AssertionError: Checkpoint-184 does not contain model weights.

In [ ]:
# Cell 24 — Diagnose the CTC blank and special-token predictions

from collections import Counter


def token_name(token_id):
    if token_id is None:
        return None
    return tokenizer.convert_ids_to_tokens(int(token_id))


special_ids = {
    "model CTC blank": model.config.pad_token_id,
    "tokenizer PAD": tokenizer.pad_token_id,
    "tokenizer UNK": tokenizer.unk_token_id,
    "tokenizer BOS": tokenizer.bos_token_id,
    "tokenizer EOS": tokenizer.eos_token_id,
    "word delimiter": tokenizer.word_delimiter_token_id,
}

for name, token_id in special_ids.items():
    print(
        f"{name}: id={token_id}, "
        f"token={token_name(token_id)!r}"
    )

print(
    "\n[UNK] occurrences in training references:",
    sum(
        example["labels"].count(tokenizer.unk_token_id)
        for example in train_ds
    ),
)

print(
    "[UNK] occurrences in validation references:",
    sum(
        example["labels"].count(tokenizer.unk_token_id)
        for example in val_clean_ds
    ),
)

frame_counts = Counter()
first_prediction_ids = None

for index in range(min(10, len(val_clean_ds))):
    example = val_clean_ds[index]

    input_values = torch.tensor(
        example["input_values"],
        dtype=torch.float32,
    ).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = model(input_values=input_values).logits

    prediction_ids = torch.argmax(
        logits,
        dim=-1,
    )[0].cpu().tolist()

    frame_counts.update(prediction_ids)

    if first_prediction_ids is None:
        first_prediction_ids = prediction_ids

total_frames = sum(frame_counts.values())

print("\nMost frequent frame-level predictions:")

for token_id, count in frame_counts.most_common(12):
    print(
        f"id={token_id:2d} "
        f"token={token_name(token_id)!r} "
        f"frames={count:6d} "
        f"percentage={100 * count / total_frames:6.2f}%"
    )

print("\nFirst prediction with special tokens:")
print(
    tokenizer.decode(
        first_prediction_ids,
        group_tokens=True,
        skip_special_tokens=False,
    )
)

print("\nFirst prediction after removing special tokens:")
print(
    tokenizer.decode(
        first_prediction_ids,
        group_tokens=True,
        skip_special_tokens=True,
    )
)